In [ ]:


import sys
import os
import psycopg2 

import sys

from pgvector.psycopg2 import register_vector

database_path = os.path.abspath("./database")

models_path = os.path.abspath("./models")

# 2. Add it to Python's system path
sys.path.append( models_path)
sys.path.append(database_path)

# 3. Now you can import it normally
import embedding_model


In [119]:
query_embedding =embedding_model. EmbeddingModel()
query_result = query_embedding.embed(
    "I need a lightweight laptop for programming with excellent battery life for a price  less than 1000"
)
type(query_result)


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 7160.72it/s]


list

In [120]:
conn = psycopg2.connect(
        dbname="smartshop",
        user="smartshop",
        password="smartshop",
        host="localhost",
        port=5432
    )
cursor = conn.cursor()
register_vector(conn)


In [121]:
query  = """SELECT
    product_id,
    content,
    1 - (embedding <=> %s::vector)  AS similarity
FROM product_embeddings
ORDER BY embedding <=> %s::vector
LIMIT %s ;"""


In [ ]:
conn.rollback()


In [ ]:
ex_result = cursor.execute(query,(query_result,query_result,10))

results = cursor.fetchall()
print(results)


[('LP1089', 'Product: Ultra Thin v84901\n    Brand: UltraComp\n    Category: laptop\n    Description:\n    15.6 inch laptop featuring long battery life.\n    Price:\n    500.4', 0.6081848325784457), ('LP1297', 'Product: Ultra Thin v96211\n    Brand: TechCo\n    Category: laptop\n    Description:\n    15.6 inch laptop featuring long battery life.\n    Price:\n    1109.08', 0.6060595512390137), ('LP1314', 'Product: Ultra Thin v92872\n    Brand: UltraComp\n    Category: laptop\n    Description:\n    13.3 inch laptop featuring long battery life.\n    Price:\n    516.09', 0.6011208891868591), ('LP0543', 'Product: Ultra Thin v55840\n    Brand: UltraComp\n    Category: laptop\n    Description:\n    15.6 inch laptop featuring long battery life.\n    Price:\n    1260.29', 0.596752405166626), ('LP1817', 'Product: Ultra Thin v8324\n    Brand: TechCo\n    Category: laptop\n    Description:\n    14 inch laptop featuring long battery life.\n    Price:\n    1154.12', 0.5957959890365601), ('LP0797', '

In [ ]:



def search_products(
    connection: Connection =conn,
    model: EmbeddingModel = embedding_model.EmbeddingModel(),
    query: str= "",
    limit: int = 5
):

    query_embedding = model.embed(query)

    with connection.cursor() as cursor:

        cursor.execute("""
            SELECT
                product_id,
                name,
                brand,
                category,
                price,
                rating,

                1 - (
                    embedding
                    <=>
                    %s
                ) AS similarity

            FROM product_embeddings 


            ORDER BY
                embedding <=> %s

            LIMIT %s
        """, (
            query_embedding,
            query_embedding,
            limit
        ))

        return cursor.fetchall()


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6299.77it/s]


In [137]:

def main():

    model = embedding_model. EmbeddingModel()

    connection = conn

    query = """
        I need a lightweight laptop for programming
        with excellent battery life
        """

    results = search_products(
            connection,
            model,
           query= query
        )

    print("\nSearch results:\n")

    for result in results:

            (
                product_id,
                name,
                brand,
                category,
                price,
                rating,
                similarity
            ) = result

            print(
                f"{name} | "
                f"{brand} | "
                f"${price} | "
                f"rating={rating} | "
                f"similarity={similarity:.4f}"
            )


if __name__ == "__main__":
    main()


Loading weights: 100%|██████████| 103/103 [00:00<00:00, 10138.78it/s]


InFailedSqlTransaction: current transaction is aborted, commands ignored until end of transaction block
